In [1]:
# Baseline data set based on decisions made in Feature-Engineering notebook

In [2]:
from data.cleaning import read_csv
from core import Config
import pandas as pd
import numpy as np

config = Config()
TARGET = config.TARGET

static_df: pd.DataFrame = read_csv(
    config.selected_dir / "selected_static.csv",
    config.selected_dir / "selected_static_dtypes.csv",
    [0]
)

historic_df: pd.DataFrame = read_csv(
    config.selected_dir / "selected_historic.csv",
    config.selected_dir / "selected_historic_dtypes.csv",
    [0]
)


## Convert and Remove Features with constant values
# Filter and convert historic dataframe
# Detect columns with only boolean-like strings and convert dtype to boolean
def is_bool_string_col(series):
    bool_strings = [{'true', 'false'}, {'True', 'False'}, {'yes', 'no'}, {'Yes', 'No'}, {'1', '0'}]
    values = set(series.dropna().unique())
    return any(values <= s for s in bool_strings)


def transform_bool_strings(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    for col in df.select_dtypes(include=['object', 'string']):
        if is_bool_string_col(df[col]):
            df[col] = df[col].replace(
                {'true': True, 'True': True, 'yes': True, 'Yes': True, '1': True,
                 'false': False, 'False': False, 'no': False, 'No': False, '0': False}).astype('boolean')
            df[col] = df[col].astype('boolean')
    return df


def convert_to_categorical(dataframe: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = dataframe.copy()
    for col in columns:
        df[col] = df[col].astype('category')
    return df


# remove columns with only one unique (non‑NaN) value
def remove_constant_columns(dataframe: pd.DataFrame) -> pd.DataFrame:
    df = dataframe.copy()
    nunique = df.nunique(dropna=True)
    cols_to_drop = nunique[nunique == 1].index
    return df.drop(columns=cols_to_drop)


historic_df = transform_bool_strings(historic_df)

# Join static and historic dataframe to have GICS Sector Codes in the dataframe
all_df = historic_df.merge(
    static_df,
    on='Instrument',
    how='left'
)

all_df = convert_to_categorical(all_df, ['TR.GICSSubIndustryCode', 'TR.GICSIndustryCode', 'TR.GICSIndustryGroupCode',
                                         'TR.GICSSectorCode', 'Date'])
all_df = remove_constant_columns(all_df)

# Manually remove redundant data
all_df.drop(
    [
        'TR.HeadquartersCountry',
        'TR.BusinessSector',
        'TR.TRBCActivity',
        'TR.TRBCIndustry',
        'TR.NAICSSector',
        'TR.NACEClassification',
        'TR.GICSSubIndustryCode',
        'TR.GICSIndustryCode',
        'TR.GICSIndustryGroupCode',
    ], axis=1, inplace=True)

"""
TR.COGSActValue & TR.F.COGSTot by definition, are relatively similar and at first glance seem to behave the same. TR.F.COGSTot has lower missingness.
TR.GPMActValue & TR.F.GrossProfMarg by definition, are relatively similar. TR.F.GrossProfMarg has lower missingness.
TR.GrossIncomeActValue & TR.F.GrossProfIndPropTot by definition, are relatively similar. TR.F.GrossProfIndPropTot has lower missingness.
TR.CO2EquivalentEmissionIndirectScope2Locationbased has almost half of its values NA. And those values both report are almost the same.
TR.HeadquartersCountry & TR.HQCountryCode are basically the same. More interesting would be in which countries a company is operating.
TR.GICSIndustryCode is the main company differentiator. All others can be looked into if time allows.
"""

# Create labeled DataFrame with row filtered emtpy Scope3.1
labeled_df: pd.DataFrame = all_df.dropna(subset=[TARGET])


# calculate missingness untouched labeled dataframe
def over_missingness(series: pd.Series, threshold=50) -> list:
    candidates: list[str] = series[series > threshold].index.tolist()
    return candidates


missing_counts: pd.Series = labeled_df.isna().sum()
percent_missing_sum_method: pd.Series = (missing_counts / labeled_df.shape[0]) * 100

# low missingness DataFrame
labeled_df.drop(over_missingness(percent_missing_sum_method), axis=1, inplace=True)

del missing_counts, percent_missing_sum_method

# Data is distributed very heterogen. In IndustryGroup the fewest category has around 300 companies, whereas in Sector over 500.

"""
TR.F.PPEGrossTot, TR.F.PPEAccumDeprTot & TR.F.PPENetTot correlate due to their dependence on company property. Gross − Accumulated Depreciation = Net. Keeping TR.F.PPENetTot, as it combines the two other values. TR.F.PPEGrossTot could also be interesting, cause it is the easiest variable to report.

TR.COGSActValue & TR.F.COGSTot highly correlate almost in the same way. Direct cost to produce vs. all factored cost to produce. Keeping TR.COGSActValue because the direct costs seem easier to report.

TR.F.TotCap, TR.NetProfitActValue, TR.EBITActValue & TR.EBITDAActValue correlate. Features report how much financial success a company has. Revenue before or after taxes, the capital, etc. Keeping TR.EBITDAActValue, no reason.

TR.TotalAssetsActual & TR.F.TotLiab highly correlate. Keeping TR.TotalAssetsActual cause it just reports what the company owns, excluding dept, etc.

TR.F.OpExpn & TR.OperatingExpActual moderately correlate (0,76). Keeping TR.F.OpExpn, because it is easier to report due to not having to cleanup the data.

TR.EBITDAActValue is the result of TR.RevenueActValue - TR.COGSActValue - TR.F.OpExpn. Company Result before Taxes ≈ Revenue - Direct Costs - Operating Expenses. Remove TR.EBITDAActValue.

TR.RevenueActValue & TR.COGSActValue highly correlate (0.91). Keeping TR.RevenueActValue, because it has lower missingness.

TR.RevenueActValue & TR.F.OpExpn highly correlate (0.97). Keeping TR.F.OpExpn, because it has lower missingness.
"""

labeled_df.drop(
    columns=[
        'TR.F.PPEGrossTot',
        'TR.F.PPEAccumDeprTot',
        'TR.F.COGSTot',
        'TR.EBITActValue',
        'TR.NetProfitActValue',
        'TR.F.TotCap',
        'TR.EBITDAActValue',
        'TR.OperatingExpActual',
        'TR.F.TotLiab',
        'TR.RevenueActValue',
        'TR.COGSActValue'], inplace=True)
labeled_df.to_csv(config.training_dir / "baseline.csv")

In [3]:
# Log transform target
log_target_df: pd.DataFrame = labeled_df.copy()
log_target_df[TARGET] = np.log1p(log_target_df[TARGET])
log_target_df.to_csv(config.training_dir / "baseline_log_target.csv")

In [12]:
# Log transform every positive numeric column
# Sign of negative numbers is preserved
log_df: pd.DataFrame = labeled_df.copy()
neg_cols: list[str] = []
for col in log_df.select_dtypes(include=['Float64', 'Int64']).columns:
    if (log_df[col] >= 0).all():
        log_df[col] = np.log1p(log_df[col])
    else:
        neg_cols.append(col)
        log_df[col] = log_df[col].transform(lambda x: np.sign(x) * (np.log1p(abs(x))))

log_df.to_csv(config.training_dir / "baseline_log_all.csv")

In [14]:
# one hot encode categorical columns
one_hot_df: pd.DataFrame = pd.get_dummies(
    labeled_df,
    columns=[
        "TR.GICSSectorCode", "TR.HQCountryCode"
    ]
)
one_hot_df.to_csv(config.training_dir / "baseline_one_hot.csv")